In [ ]:
from datetime import datetime, timedelta
import json
from pathlib import Path

from matplotlib.dates import DateFormatter
import matplotlib.pyplot as plt

import numpy as np
import polars as pl

from scipy.stats import boxcox
from scipy.optimize import minimize

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, ARIMA

## Dynamic Regression Baseline Model

In [ ]:
class BoxCoxScaler:
    def __init__(self):
        self._lambda: float | None = None

    @property
    def is_fit(self):
        return self._lambda is not None
    
    def fit_transform(self, y: pl.Series) -> pl.Series:
        y_t, _lambda = boxcox(y.to_numpy())
        self._lambda = _lambda
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit
        y_t = boxcox(y.to_numpy(), lmbda=self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def inverse_transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit

        if self._lambda == 0:
            y_t = np.exp(y.to_numpy())
        else:
            y_t = (y.to_numpy() * self._lambda + 1) ** (1 / self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)


class CyclicFeatureTransformer:
    def __init__(self, max_harmonic: int = 5):
        self.max_harmonic = max_harmonic

    def transform(self, df: pl.DataFrame) -> pl.DataFrame:
        # Fourier cycles
        fourier_cycles = [
            ("hour", pl.col("timestamp").dt.hour(), 24),
            ("day", pl.col("timestamp").dt.day(), 7),
            ("month", pl.col("timestamp").dt.month(), 12),
        ]
        col_expr = {}
        for unit, t_expr, period in fourier_cycles:
            for k in range(1, self.max_harmonic + 1):
                angle = (2 * np.pi * k * t_expr) / period
                col_expr[f"sin_{k}_{unit}"] = np.sin(angle)
                col_expr[f"cos_{k}_{unit}"] = np.cos(angle)
        
        # Weekend indicator
        col_expr["is_weekend"] = (pl.col("timestamp").dt.weekday() >= 6).cast(pl.Float32)
        
        return df.with_columns(**col_expr)

    def get_feature_names(self) -> list[str]:
        cyclic_features = [
            f"{prefix}_{k}_{unit}"
            for unit in ["hour", "day", "month"]
            for k in range(1, self.max_harmonic + 1)
            for prefix in ["sin", "cos"]
        ]
        return cyclic_features + ["is_weekend"]


class LinearRegressionModel:
    def __init__(self, lasso: float = 0.0, include_bias: bool = True):
        self.lasso = lasso
        self.include_bias = include_bias


    def fit(self, X: np.ndarray, y: np.ndarray):
        if self.include_bias:
            X = np.hstack([np.ones((X.shape[0], 1)), X])

        # Smooth L1 approximation: sqrt(theta^2 + eps) ≈ |theta|, differentiable at 0
        _eps = 1e-8

        def f_(theta: np.ndarray):
            y_hat = np.dot(X, theta)
            l1_smooth = np.sum(np.sqrt(theta**2 + _eps))
            return np.mean((y - y_hat) ** 2) + self.lasso * l1_smooth

        x0 = np.array([0.5 for _ in range(X.shape[1])])
        result = minimize(f_, x0=x0, method="L-BFGS-B", tol=1e-6, options={"maxiter": 5000})
        if not result.success:
            raise ValueError(result.message)
        self.theta_ = result.x

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.include_bias:
            X = np.hstack([np.ones((X.shape[0], 1)), X])
        return np.dot(X, self.theta_)


class DynamicRegressionModel:
    def __init__(
        self,
        order: tuple[int, int, int],
        seasonal_order: tuple[int, int, int, int],
        include_bias: bool = True,
        lasso: float = 0.0,
    ):
        self.include_bias = include_bias
        self.order = order
        self.seasonal_order = seasonal_order
        self.lasso = lasso

        self._lr_model: LinearRegressionModel | None = None
        self._arima_model: ARIMA | None = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        self._lr_model = LinearRegressionModel(include_bias=self.include_bias, lasso=self.lasso)
        self._lr_model.fit(X, y)

        lr_y_hat = self._lr_model.predict(X)
        y_residuals = y - lr_y_hat

        self._arima_model = ARIMA(
            order=self.order,
            season_length=24,
            seasonal_order=self.seasonal_order,
            include_mean=True,
            include_drift=False,
        )
        self._arima_model = self._arima_model.fit(y_residuals)

        return self

    def predict_in_sample(self, X: np.ndarray):
        lr_y_hat = self._lr_model.predict(X)
        arima_y_hat = self._arima_model.predict_in_sample()
        return lr_y_hat + arima_y_hat["fitted"]

    def predict(self, X: np.ndarray):
        horizon = X.shape[0]
        lr_y_hat = self._lr_model.predict(X)
        arima_y_hat = self._arima_model.predict(h=horizon)
        y_hat = lr_y_hat + arima_y_hat["mean"]

        return y_hat

## PJM Dataset

In [ ]:
PJM_SITE_NAME = "PJMW"
PJM_DATA_FREQUENCY = "1h"

INPUT_PATH = Path("../../data/pjm")
OUTPUT_PATH = Path(f"../../results/pjm/naive/{PJM_SITE_NAME}")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

data_file_name = f"{PJM_SITE_NAME}_hourly_processed.pq"
data_file_path = INPUT_PATH / data_file_name
SITE_DF = pl.read_parquet(data_file_path).sort(by="timestamp")

### Configure

In [ ]:
MAX_FOURIER_HARMONIC = 5
TIMESTAMP_COL = "timestamp"
TARGET_COL = f"{PJM_SITE_NAME}_MW"
SCALED_TARGET_COL = f"{TARGET_COL}_SCALED"

### Fourier Regression Model

In [ ]:
fft = CyclicFeatureTransformer(max_harmonic=MAX_FOURIER_HARMONIC)
scaler = BoxCoxScaler()

train_df = SITE_DF.clone()

X = fft.transform(train_df)
X = X.select(pl.col(fft.get_feature_names()))

y = train_df[TARGET_COL]
y_scaled = scaler.fit_transform(y)

model = LinearRegressionModel(include_bias=True, lasso=0.05)
model = model.fit(X.to_numpy(), y_scaled.to_numpy())
y_hat = model.predict(X.to_numpy())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)

n_samples = 1000
axes[0].plot(y_scaled[:n_samples])
axes[0].plot(y_hat[:n_samples])
axes[0].set(title="Fourier Regression Model", ylabel="MW (scaled)")

residuals = (y_scaled - y_hat)
axes[1].plot(residuals[:n_samples])
axes[1].axhline(0, color="black", ls="--", lw=0.5, alpha=0.5)
axes[1].set(ylabel="Residuals")

seasonal_diff_order = 24
residuals_diff = (residuals[seasonal_diff_order:] - residuals[:-seasonal_diff_order]).to_numpy()
residuals_diff = residuals_diff[1:] - residuals_diff[:-1]

axes[2].plot(residuals_diff[:n_samples])
axes[2].axhline(0, color="black", ls="--", lw=0.5, alpha=0.5)
axes[2].set(xlabel="Timestamp", ylabel="Residuals (diff)")

fig.tight_layout();

In [ ]:
# Plot autocorrelation of residuals and residuals_diff
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)

diff_order = 24
plot_acf(residuals.to_numpy(), ax=axes[0])

seasonal_diff_order = 24
residuals_diff = (residuals[seasonal_diff_order:] - residuals[:-seasonal_diff_order]).to_numpy()
residuals_diff = residuals_diff[1:] - residuals_diff[:-1]
plot_acf(residuals_diff, ax=axes[1])
plot_pacf(residuals_diff, ax=axes[2])

fig.tight_layout();

### Fourier Regression Model with ARIMA Errors

In [ ]:
fft = CyclicFeatureTransformer(max_harmonic=MAX_FOURIER_HARMONIC)
scaler = BoxCoxScaler()

n_test_samples = 1000
train_df = SITE_DF.slice(0, len(SITE_DF) - n_test_samples)
valid_df = SITE_DF.slice(-n_test_samples)

X_train = fft.transform(train_df)
X_train = X_train.select(pl.col(fft.get_feature_names()))

y_train = train_df[TARGET_COL]
y_train_scaled = scaler.fit_transform(y_train)

X_valid = fft.transform(valid_df)
X_valid = X_valid.select(pl.col(fft.get_feature_names()))

y_valid = valid_df[TARGET_COL]
y_valid_scaled = scaler.transform(y_valid)

In [ ]:
model = DynamicRegressionModel(
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1),
    include_bias=True,
    lasso=0.05,
)
model = model.fit(X_train.to_numpy(), y_train_scaled.to_numpy())

# Get fitted values
y_fitted = model.predict_in_sample(X_train.to_numpy())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(y_train_scaled[-1000:])
axes[0].plot(y_fitted[-1000:])
axes[0].set(title="Dynamic Regression Model - Fitted Values", ylabel="MW (scaled)")

# Residuals
residuals = y_train_scaled - y_fitted
axes[1].plot(residuals[:1000])
axes[1].axhline(0, color="black", ls="--", lw=0.5, alpha=0.5)
axes[1].set(ylabel="Residuals")

fig.tight_layout();

In [ ]:
# Get forecasts
y_hat = model.predict(X_valid.to_numpy())
y_hat = pl.Series(name=f"{SCALED_TARGET_COL}_FORECAST", values=y_hat)
y_hat_scaled = scaler.inverse_transform(y_hat)

fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))
ax.plot(y_valid[:1000])
ax.plot(y_hat_scaled[:1000])
ax.set(title="Dynamic Regression Model - Forecasts", ylabel="MW")
fig.tight_layout();